# Lesson 5.5 — Debugging, Error Handling, and Trace Analysis

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Identify error types (API, rate limit, tool, model-output)
- Choose recovery strategies (retry/backoff, fallback, abort)
- Analyze a trace to localize a failure
- Isolate integration-layer vs model-output failures

**You will produce:** A retry-with-backoff wrapper and a trace analyzer that classifies failure origin.

## Concepts

Robust Claude apps distinguish **where** a failure originates (Domain 4):

- **Integration-layer failures**: network errors, 429 rate limits, auth errors, malformed requests, tool exceptions. Fix in *your* code/config.
- **Model-output failures**: valid call but wrong/invalid content (bad JSON, hallucination, missed constraint). Fix with prompts, schemas, validation, or re-asking.

**Recovery strategies**:
- **Retry with exponential backoff** for transient errors (timeouts, 429, 5xx).
- **Fallback** (smaller model, cached answer, default) when retries fail.
- **Abort/escalate** for non-retryable errors (auth, invalid request).

**Trace analysis**: log each step (request, stop_reason, tool calls, usage, errors) so you can localize the failing stage.

> **Reference:** [Anthropic - Errors & handling](https://docs.anthropic.com/en/api/errors)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Classify an error's origin

Map an exception/result to integration vs model-output.

In [ ]:
def classify_failure(kind):
    integration = {"timeout", "rate_limit", "auth", "network", "tool_exception"}
    model_output = {"bad_json", "schema_violation", "hallucination", "constraint_miss"}
    if kind in integration:
        return "integration-layer (fix your code/config)"
    if kind in model_output:
        return "model-output (fix prompt/schema/validation)"
    return "unknown - gather more trace detail"

for k in ["rate_limit", "bad_json", "auth", "hallucination"]:
    print(k, "->", classify_failure(k))

## Code-Along: Step 2 — Retry with exponential backoff

Retry transient errors; give up cleanly on non-retryable ones.

In [ ]:
import time, random

RETRYABLE = {"timeout", "rate_limit", "network", "server_error"}

def with_retry(call, classify, max_tries=4, base=0.5):
    for attempt in range(max_tries):
        try:
            return call()
        except Exception as e:
            kind = classify(e)
            if kind not in RETRYABLE or attempt == max_tries - 1:
                return {"error": f"giving up: {kind}"}
            delay = base * (2 ** attempt) + random.random() * 0.1
            print(f"retry {attempt+1} after {delay:.2f}s ({kind})")
            time.sleep(min(delay, 0.2))  # capped for the demo

attempts = {"n": 0}
def flaky():
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise TimeoutError("slow")
    return "ok"

print(with_retry(flaky, classify=lambda e: "timeout"))

## Code-Along: Step 3 — Analyze a trace

Walk a recorded trace and flag the first failing stage.

In [ ]:
trace = [
    {"stage": "request", "ok": True},
    {"stage": "tool_call:search", "ok": True},
    {"stage": "parse_output", "ok": False, "error": "bad_json"},
]

def analyze(trace):
    for step in trace:
        if not step["ok"]:
            return {"failed_stage": step["stage"],
                    "origin": classify_failure(step.get("error", ""))}
    return {"failed_stage": None}

print(analyze(trace))

## Exercise

Extend `with_retry` to accept a `fallback` callable used when all retries fail (e.g., call a cheaper model or return a cached answer). Demonstrate it firing after exhausting retries.

In [ ]:
def with_retry_fallback(call, classify, fallback, max_tries=3):
    # TODO: reuse retry logic; on final failure, return fallback()
    pass

# TODO: show fallback() returning a cached/default answer after retries fail

## Key Takeaways

- Distinguish integration-layer failures (your code) from model-output failures (prompt/schema).
- Retry transient errors with exponential backoff; abort on non-retryable ones.
- Add fallbacks (cheaper model, cache, default) for graceful degradation.
- Log structured traces so you can localize the failing stage quickly.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.